# Solution: Simple PID Controller


In [11]:
# Import standard libraries
import math
from pathlib import Path
import time

# Import third-party libraries
from IPython.display import clear_output
import mujoco
import mujoco.viewer
import numpy as np

In [12]:
# Settings
MJCF_PATH = Path("../../mechanical/bala-c-plus-simplified/bala-c-plus-simplified.xml")
MOTOR_SPEED_LIMIT = 1.0    # Max motor speed in each direction
PRINT_EVERY = 50           # Number of sim loop iterations before printing sensor readings

# Actuator names (from MJCF file)
LEFT_MOTOR = "left_motor"
RIGHT_MOTOR = "right_motor"

# Sensor names (from MJCF file)
IMU_ACCEL = "imu_accel"
IMU_GYRO = "imu_gyro"
IMU_ORIENTATION = "imu_orientation"

In [13]:
def clamp(x):
    """Limit motor speed and direction to a minimum and maximum"""
    return max(-MOTOR_SPEED_LIMIT, min(MOTOR_SPEED_LIMIT, x))

In [14]:
# Load model into MuJoCo
model = mujoco.MjModel.from_xml_path(str(MJCF_PATH))

# Use model to get the simulation state
data  = mujoco.MjData(model)

In [15]:
# Get ID of actuators from MJCF names
left_motor_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_ACTUATOR, LEFT_MOTOR)
right_motor_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_ACTUATOR, RIGHT_MOTOR)

# Print IDs
print(f"Left motor ID: {left_motor_id}")
print(f"Right motor ID: {right_motor_id}")

Left motor ID: 0
Right motor ID: 1


In [17]:
# Adjust that arange to figure out where the bot balances most naturally
for deg in np.arange(-15, 16, 1.0):
    th = math.radians(deg)
    mujoco.mj_resetData(model, data)
    data.qpos[3:7] = [math.cos(th/2), 0, math.sin(th/2), 0]
    data.qvel[:] = 0
    mujoco.mj_forward(model, data)
    p0 = deg
    for _ in range(100):
        data.ctrl[:] = 0
        mujoco.mj_step(model, data)
    w,x,y,z = data.sensor("imu_orientation").data
    p1 = math.degrees(math.atan2(1-2*(x*x+y*y), -2*(y*z+w*x)))
    print(f"start {deg:.2f} -> after 100 steps {p1:+.2f} (moved {abs(p1-abs(p0)):.2f})")

start -15.00 -> after 100 steps +105.52 (moved 90.52)
start -14.00 -> after 100 steps +105.52 (moved 91.52)
start -13.00 -> after 100 steps +105.53 (moved 92.53)
start -12.00 -> after 100 steps +105.53 (moved 93.53)
start -11.00 -> after 100 steps +105.53 (moved 94.53)
start -10.00 -> after 100 steps +105.53 (moved 95.53)
start -9.00 -> after 100 steps +105.52 (moved 96.52)
start -8.00 -> after 100 steps +105.53 (moved 97.53)
start -7.00 -> after 100 steps +105.54 (moved 98.54)
start -6.00 -> after 100 steps +105.54 (moved 99.54)
start -5.00 -> after 100 steps +105.52 (moved 100.52)
start -4.00 -> after 100 steps +105.48 (moved 101.48)
start -3.00 -> after 100 steps +105.55 (moved 102.55)
start -2.00 -> after 100 steps +105.58 (moved 103.58)
start -1.00 -> after 100 steps +105.52 (moved 104.52)
start 0.00 -> after 100 steps +105.63 (moved 105.63)
start 1.00 -> after 100 steps +105.61 (moved 104.61)
start 2.00 -> after 100 steps +105.62 (moved 103.62)
start 3.00 -> after 100 steps +105.

In [55]:
# Trim angle: true angle the bot needs to balance at (with offset CoM)
PITCH_TRIM = math.radians(11.4)

g = float(np.linalg.norm(model.opt.gravity))

def read(d):
    _, accel_y, accel_z = d.sensor("imu_accel").data
    w,x,y,z = d.sensor("imu_orientation").data
    pitch = math.atan2(1.0-2.0*(x*x+y*y), -2.0*(y*z+w*x))
    return accel_y, accel_z, pitch

# Hold at trim lean, zero control, let contact settle, then read
mujoco.mj_resetData(model, data)
th = PITCH_TRIM
data.qpos[3:7] = [math.cos(th/2), 0, math.sin(th/2), 0]
data.qvel[:] = 0
mujoco.mj_forward(model, data)
for _ in range(50):                      # let it settle on its wheels
    data.ctrl[:] = 0
    mujoco.mj_step(model, data)

accel_y, accel_z, pitch = read(data)
a_fwd = accel_y - g * math.sin(pitch)    # the estimator's forward-accel term

print(f"at trim lean ({math.degrees(pitch):.1f} deg), stationary:")
print(f"  accel_y      = {accel_y:+.4f}")
print(f"  accel_z      = {accel_z:+.4f}")
print(f"  g*sin(pitch) = {g*math.sin(pitch):+.4f}")
print(f"  a_fwd        = {a_fwd:+.4f}   <-- want ~0")

at trim lean (-8.7 deg), stationary:
  accel_y      = -9.6823
  accel_z      = -1.7461
  g*sin(pitch) = -1.4866
  a_fwd        = -8.1957   <-- want ~0


In [ ]:
# Filter and PID coefficients (tune these)
KP = 20.0
KD = 0.5

# Trim angle: true angle the bot needs to balance at (with offset CoM)
PITCH_TRIM = math.radians(11.4)

# When the robot has "tipped over" into an unrecoverable state
TIP_THRESHOLD = math.radians(30)

# Resets simulation data to defaults
mujoco.mj_resetData(model, data)

# Start at the equilibrium lean
data.qpos[3:7] = [math.cos(PITCH_TRIM/2), 0, math.sin(PITCH_TRIM/2), 0]
mujoco.mj_forward(model, data)

# Get gravity and timestep from sim
g = float(np.linalg.norm(model.opt.gravity))
dt = model.opt.timestep

# Whatever pitch_of reads here IS the controller's target — no sign guessing.
w,x,y,z = data.sensor(IMU_ORIENTATION).data
PITCH_TRIM = math.atan2(1.0-2.0*(x*x+y*y), -2.0*(y*z+w*x))
print(f"equilibrium reads as pitch = {math.degrees(PITCH_TRIM):+.2f} deg")

# Launch MuJoCo simulator and GUI
vel_est = 0.0
steps = 0
with mujoco.viewer.launch_passive(model, data) as viewer:
    # Define free-look camera (control with mouse), looking at robot's back-right
    viewer.cam.type = mujoco.mjtCamera.mjCAMERA_FREE
    viewer.cam.lookat[:] = [0, 0, 0.05]
    viewer.cam.distance  = 0.8 
    viewer.cam.azimuth   = 45
    viewer.cam.elevation = -25

    # Simulation loop
    while viewer.is_running():
        step_start = time.time()

        w, x, y, z = data.sensor(IMU_ORIENTATION).data
        pitch = math.atan2(1.0 - 2.0*(x*x+y*y), -2.0*(y*z+w*x))
        pitch_rate = data.sensor(IMU_GYRO).data[0]

        # Calculate forward acceleration
        _, accel_y, accel_z = data.sensor(IMU_ACCEL).data
        a_fwd = accel_z - g * math.sin(pitch)

        # Leaky integrator to estimate velocity
        vel_est = 0.995 * (vel_est + a_fwd * dt)
    
        ctrl = -(KP*(pitch - PITCH_TRIM) + KD*pitch_rate)
        ctrl = max(-1.0, min(1.0, ctrl))
        data.ctrl[left_motor_id]  = ctrl
        data.ctrl[right_motor_id] = ctrl
        mujoco.mj_step(model, data)
    
        if steps % 20 == 0:
            wv = data.sensor("left_wheel_vel").data[0]
            true_vel = wv * 0.03325
            print(f"t={steps:4d}  pitch={math.degrees(pitch):+6.1f}  a_fwd={a_fwd:+6.2f}  "
                  f"vel_est={vel_est:+6.3f}  true_vel={true_vel:+6.3f}  wheel={wv:+.2f}")

        if abs(pitch) > math.radians(45):
            continue
            print(f"tipped at step {steps}");

        # Render the current simulation state
        viewer.sync()

        # Just print what the sensors say when the robot is upright
        viewer.sync()
        slack = dt - (time.time() - step_start)
        if slack > 0:
            time.sleep(slack)
        steps += 1

equilibrium reads as pitch = -11.40 deg
t=   0  pitch= -11.4  a_fwd= +1.94  vel_est=+0.010  true_vel=+0.000  wheel=+0.00
t=  20  pitch= -11.1  a_fwd= -0.04  vel_est=+0.001  true_vel=-0.003  wheel=-0.09
t=  40  pitch= -11.0  a_fwd= -0.09  vel_est=-0.007  true_vel=-0.016  wheel=-0.47
t=  60  pitch= -10.9  a_fwd= -0.09  vel_est=-0.015  true_vel=-0.025  wheel=-0.76
t=  80  pitch= -10.8  a_fwd= -0.10  vel_est=-0.022  true_vel=-0.035  wheel=-1.06
t= 100  pitch= -10.7  a_fwd= -0.10  vel_est=-0.029  true_vel=-0.045  wheel=-1.36
t= 120  pitch= -10.7  a_fwd= -0.11  vel_est=-0.036  true_vel=-0.056  wheel=-1.69
t= 140  pitch= -10.6  a_fwd= -0.11  vel_est=-0.043  true_vel=-0.067  wheel=-2.02
t= 160  pitch= -10.5  a_fwd= -0.11  vel_est=-0.050  true_vel=-0.079  wheel=-2.37
t= 180  pitch= -10.4  a_fwd= -0.12  vel_est=-0.056  true_vel=-0.091  wheel=-2.74
t= 200  pitch= -10.3  a_fwd= -0.12  vel_est=-0.062  true_vel=-0.104  wheel=-3.12
t= 220  pitch= -10.2  a_fwd= -0.13  vel_est=-0.068  true_vel=-0.117  